In [1]:
import os
import time
import random
import uuid
from typing import List, Dict, Any, Optional

from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI

In [2]:
# Initial configurations and API settings
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.avalai.ir/v1")

INDEX_NAME = "amnesia-test-memory"
EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o"
EMBEDDING_DIMENSION = 1536

# Initialize clients
client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
pc = Pinecone(api_key=PINECONE_API_KEY)

In [ ]:
# Helper functions
def get_embedding_with_retry(text: str, max_retries: int = 3) -> Optional[List[float]]:
    """Get embeddings with exponential backoff for rate limits."""
    for attempt in range(max_retries):
        try:
            response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
            return response.data[0].embedding
        except Exception as e:
            error_message = str(e).lower()
            if "429" in error_message or "rate" in error_message:
                wait_time = (2 ** attempt) + random.uniform(0, 1)
                print(f"   ⏳ API Limit! Waiting {wait_time:.1f} seconds... (Attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"❌ Unexpected error in embedding: {e}")
                return None
    print(f"❌ Failed to get embedding after {max_retries} attempts.")
    return None

def setup_pinecone() -> Any:
    """Initialize and return the Pinecone index."""
    print("⏳ Checking and initializing Pinecone...")
    print("*" * 80)
    
    existing_indexes = pc.list_indexes().names()
    if INDEX_NAME not in existing_indexes:
        print(f"🔨 Creating new Index '{INDEX_NAME}'...")
        pc.create_index(
            name=INDEX_NAME,
            dimension=EMBEDDING_DIMENSION,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        time.sleep(5) # Wait for index to be ready
    return pc.Index(INDEX_NAME)

# Part 1: Stateless Chat (No Memory)
def stateless_chat() -> None:
    print("\n🧠 Test 1: Stateless Chat (Amnesia Test)")
    print("=" * 50)
    
    # First conversation: Introduction
    user_msg_1 = "سلام! من شهاب هستم. من عاشق یادگیری ماشین و برنامه‌نویسی پایتون هستم."
    print(f"👤 You: {user_msg_1}\n")
    
    response_1 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": user_msg_1}]
    )
    print(f"🤖 Bot: {response_1.choices[0].message.content}")
    print("-" * 50)
    
    # Second conversation: Amnesia test (without sending previous context)
    user_msg_2 = "اسم من چیه و به چه چیزهایی علاقه دارم؟"
    print(f"👤 You: {user_msg_2}\n")
    
    response_2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": user_msg_2}]
    )
    print(f"🤖 Bot: {response_2.choices[0].message.content}")
    print("-" * 50 + "\n")
    print("⚠️ As you can see, the bot has no idea who you are!")

# Vector Memory Class (With Namespace)
class VectorMemory:
    """Handles long-term memory operations using Pinecone with Namespace isolation."""
    def __init__(self, index: Any, namespace: str):
        self.index = index
        self.namespace = namespace # Unique namespace for each session

    def add_message(self, role: str, content: str) -> None:
        """Save message in vector database as long-term memory."""
        embedding = get_embedding_with_retry(content)
        if embedding:
            # Generate a unique ID for the message
            message_id = str(uuid.uuid4())
            self.index.upsert(
                vectors=[{
                    "id": message_id,
                    "values": embedding,
                    "metadata": {
                        "role": role,
                        "content": content,
                        "timestamp": time.time()
                    }
                }],
                namespace=self.namespace # Isolate data!
            )

    def get_context(self, query_embedding: List[float], k: int = 5) -> List[Dict[str, str]]:
        """Retrieve the most relevant past messages from this specific namespace."""
        results = self.index.query(
            vector=query_embedding,
            top_k=k,
            include_metadata=True,
            namespace=self.namespace # Search only in this session's data!
        )
        
        # Extract and format retrieved messages
        retrieved_messages = []
        for match in results.matches:
            if hasattr(match, 'metadata') and match.metadata:
                retrieved_messages.append({
                    "role": str(match.metadata["role"]),
                    "content": str(match.metadata["content"]),
                    "timestamp": float(match.metadata.get("timestamp", 0.0))
                })
        
        # Sort messages from oldest to newest to maintain logical conversation flow
        retrieved_messages.sort(key=lambda x: x["timestamp"])
        
        # Remove the timestamp field because OpenAI API only accepts 'role' and 'content'
        formatted_messages = []
        for msg in retrieved_messages:
            formatted_messages.append({
                "role": str(msg["role"]),
                "content": str(msg["content"])
            })
            
        return formatted_messages

# Part 2: Stateful Chat with Vector Memory
def stateful_chat_with_memory(index: Any) -> None:
    print("\n💾 Test 2: Stateful Chat (Vector Memory)")
    print("=" * 50)
    
    # Generate a unique session ID to avoid mixing data from previous runs
    session_id = f"session_{uuid.uuid4().hex[:8]}"
    print(f"🔗 Using Namespace for isolation: {session_id}")
    
    memory = VectorMemory(index, namespace=session_id)
    
    # First conversation: Introduction and saving to memory
    user_msg_1 = "سلام! من سارا هستم. ۲۵ سالمه و طراح گرافیک هستم."
    print(f"👤 You: {user_msg_1}\n")
    
    # Add user message to memory
    memory.add_message("user", user_msg_1)
    
    response_1 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": user_msg_1}]
    )
    bot_reply_1 = str(response_1.choices[0].message.content)
    print(f"🤖 Bot: {bot_reply_1}")
    
    # Add bot response to memory
    memory.add_message("assistant", bot_reply_1)
    
    print("⏳ Saving to Pinecone and waiting for sync... (6 seconds)")
    time.sleep(5) 
    print("-" * 50)
    
    # Second conversation: Retrieval from memory
    user_msg_2 = "میشه بگی من کی هستم و شغلم چیه؟"
    print(f"👤 You: {user_msg_2}\n")
    
    # 1. Convert the new question into a vector
    query_embedding = get_embedding_with_retry(user_msg_2)
    if not query_embedding:
        print("❌ Failed to process query embedding.")
        return
        
    # 2. Extract context (memories) from the database
    past_context = memory.get_context(query_embedding, k=5)
    
    # 3. Combine memories with the new question
    messages_for_llm: List[Dict[str, str]] = [
        {"role": "system", "content": "تو یک دستیار هوشمند هستی. از اطلاعات قبلی کاربر برای پاسخ دادن استفاده کن."}
    ]
    messages_for_llm.extend(past_context)
    messages_for_llm.append({"role": "user", "content": user_msg_2}) # Add current question
    
    response_2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages_for_llm
    )
    print(f"🤖 Bot (with memory): {response_2.choices[0].message.content}")
    print("-" * 50 + "\n")
    print("✅ As you can see, the bot has an idea of who you are!\n")

In [4]:
# Main Execution
def main() -> None:
    try:
        # Run the first test (Amnesia)
        stateless_chat()
        
        # Setup database for the second test
        index = setup_pinecone()
        
        # Run the second test (With memory)
        stateful_chat_with_memory(index)
        
    except Exception as e:
        print(f"\n❌ General Error: {e}")

if __name__ == "__main__":
    main()


🧠 Test 1: Stateless Chat (Amnesia Test)
👤 You: سلام! من شهاب هستم. من عاشق یادگیری ماشین و برنامه‌نویسی پایتون هستم.

🤖 Bot: سلام شهاب! خیلی خوشحالم که با یک علاقه‌مند به یادگیری ماشین و برنامه‌نویسی پایتون صحبت می‌کنم. هر دو این زمینه‌ها فوق‌العاده جذاب و پرچالش هستند! اگر سوال یا موضوعی درباره یادگیری ماشین یا پایتون داری که بخوای با هم بررسی کنیم، من در خدمتتم. 💻😊
--------------------------------------------------
👤 You: اسم من چیه و به چه چیزهایی علاقه دارم؟

🤖 Bot: من متأسفانه نمی‌توانم نام شما یا علایق شخصی شما را حدس بزنم، چون اطلاعات مشخصی از شما ندارم. اگر دوست دارید، می‌توانید خودتان را معرفی کنید یا در مورد علایقتان با من صحبت کنید. خوشحال می‌شوم گفت‌وگو کنیم! 😊
--------------------------------------------------

⚠️ As you can see, the bot has no idea who you are!
⏳ Checking and initializing Pinecone...
****************************************************************************************************

💾 Test 2: Stateful Chat (Vector Memory)
🔗 Using Namespace for isolation